In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
# List what is available in /kaggle/input/
for d in os.listdir('/kaggle/input/'):
 print(d)


# Then list the contents of the dataset folder
dataset_path = '/kaggle/input/datasets'
print('\nDataset contents:')
for item in os.listdir(dataset_path)[:20]: # First 20 items
 print(' ', item)


In [ ]:
%%writefile preprocess.py

"""
preprocess.py
SignBridge ML Pipeline — Video → Landmark Preprocessing

For each video clip:
  1. Extract frames at 30 FPS using OpenCV
  2. Run MediaPipe Hand Landmarker (Tasks API) on each frame
  3. Extract 21×3 = 63 float values per hand (pad to 2 hands = 126)
  4. Apply landmark normalization (Python port of C++ normalizer)
  5. Apply Kalman filter smoothing
  6. Save as .npy arrays: shape (L, 126) where L = fixed sequence length

Usage:
  python preprocess.py --input_dir data/raw/WLASL100 --output_dir data/processed
                       --sequence_length 64 --fps 30
"""

import argparse
import os
import sys
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm


# ============================================================
# Python ports of C++ core functions (for training pipeline)
# ============================================================

def normalize_landmarks_py(raw: np.ndarray) -> np.ndarray:
    """
    Normalize 21 hand landmarks (Python port of C++ normalize_landmarks).
    
    Args:
        raw: np.ndarray shape (21, 3) — raw landmarks from MediaPipe
    Returns:
        np.ndarray shape (21, 3) — normalized landmarks
    """
    out = raw.copy()
    
    # 1. Translate: subtract wrist position
    wrist = out[0].copy()
    out -= wrist
    
    # 2. Scale: divide by wrist-to-MCP9 distance
    scale = np.linalg.norm(out[9])
    if scale < 1e-6:
        scale = 1.0
    out /= scale
    
    return out


class KalmanFilter1D:
    """
    1D Kalman filter (Python port of C++ KalmanFilter1D).
    Default: Q=0.001, R=0.01, P=1.0
    """
    def __init__(self, Q=0.001, R=0.01, P=1.0):
        self.Q = Q
        self.R = R
        self.P = P
        self.x = 0.0
    
    def update(self, z: float) -> float:
        self.P = self.P + self.Q
        K = self.P / (self.P + self.R)
        self.x = self.x + K * (z - self.x)
        self.P = (1.0 - K) * self.P
        return self.x
    
    def reset(self):
        self.P = 1.0
        self.x = 0.0


class LandmarkKalmanBank:
    """Bank of 63 Kalman filters for one hand (21 landmarks × 3 axes)."""
    def __init__(self):
        self.filters = [[KalmanFilter1D() for _ in range(3)] for _ in range(21)]
    
    def update(self, landmarks: np.ndarray) -> np.ndarray:
        """
        Args:
            landmarks: shape (21, 3)
        Returns:
            smoothed: shape (21, 3)
        """
        smoothed = np.zeros_like(landmarks)
        for i in range(21):
            for j in range(3):
                smoothed[i, j] = self.filters[i][j].update(float(landmarks[i, j]))
        return smoothed
    
    def reset(self):
        for i in range(21):
            for j in range(3):
                self.filters[i][j].reset()


# ============================================================
# MediaPipe Hand Landmark Extraction
# ============================================================

def extract_landmarks_from_video(video_path: str, fps: int = 25) -> list:
    """
    Extract hand landmarks from a video file using MediaPipe.
    
    Args:
        video_path: Path to video file
        fps: Target FPS for extraction
    
    Returns:
        List of frames, each a dict with 'left' and 'right' hand landmarks
        (each np.ndarray shape (21,3) or None if hand not detected)
    """
    try:
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision
    except ImportError:
        print("WARNING: MediaPipe not available. Using dummy extraction.")
        return _extract_landmarks_dummy(video_path, fps)
    
    # Initialize Hand Landmarker
    model_path = _get_mediapipe_model_path()
    if model_path is None:
        print("WARNING: MediaPipe model not found. Using dummy extraction.")
        return _extract_landmarks_dummy(video_path, fps)
    
    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        running_mode=vision.RunningMode.VIDEO
    )
    
    landmarker = vision.HandLandmarker.create_from_options(options)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open video: {video_path}")
        return []
    
    source_fps = cap.get(cv2.CAP_PROP_FPS)
    if source_fps <= 0:
        source_fps = 25.0
    
    frame_interval = max(1, int(source_fps / fps))
    frames_data = []
    frame_idx = 0
    timestamp_ms = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx % frame_interval == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            timestamp_ms += int(1000 / fps)
            
            left_hand = None
            right_hand = None
            
            if result.hand_landmarks:
                for i, handedness in enumerate(result.handedness):
                    hand_label = handedness[0].category_name.lower()
                    landmarks = np.array([
                        [lm.x, lm.y, lm.z] 
                        for lm in result.hand_landmarks[i]
                    ], dtype=np.float32)
                    
                    if hand_label == 'left':
                        left_hand = landmarks
                    else:
                        right_hand = landmarks
            
            frames_data.append({
                'left': left_hand,
                'right': right_hand
            })
        
        frame_idx += 1
    
    cap.release()
    landmarker.close()
    return frames_data


def _get_mediapipe_model_path() -> str:
    """Find the MediaPipe hand landmarker model file."""
    candidates = [
        'hand_landmarker.task',
        'models/hand_landmarker.task',
        os.path.join(os.path.dirname(__file__), '..', 'models', 'hand_landmarker.task'),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def _extract_landmarks_dummy(video_path: str, fps: int) -> list:
    """Dummy extraction for when MediaPipe model is not available."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    source_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    duration = total_frames / source_fps
    n_frames = int(duration * fps)
    cap.release()
    
    frames_data = []
    for _ in range(max(n_frames, 1)):
        frames_data.append({
            'left': np.random.randn(21, 3).astype(np.float32) * 0.1 + 0.5,
            'right': np.random.randn(21, 3).astype(np.float32) * 0.1 + 0.5,
        })
    return frames_data


# ============================================================
# Full Preprocessing Pipeline
# ============================================================

def process_single_video(video_path: str, sequence_length: int = 64,
                         fps: int = 25) -> np.ndarray:
    """
    Full preprocessing pipeline for a single video.
    
    Returns:
        np.ndarray shape (sequence_length, 126) or None if extraction fails
    """
    # Extract landmarks
    frames_data = extract_landmarks_from_video(video_path, fps)
    if not frames_data:
        return None
    
    # Initialize Kalman banks for each hand
    kalman_left = LandmarkKalmanBank()
    kalman_right = LandmarkKalmanBank()
    
    processed_frames = []
    
    for frame in frames_data:
        # Process left hand
        if frame['left'] is not None:
            left_norm = normalize_landmarks_py(frame['left'])
            left_smooth = kalman_left.update(left_norm)
        else:
            left_smooth = np.zeros((21, 3), dtype=np.float32)
        
        # Process right hand
        if frame['right'] is not None:
            right_norm = normalize_landmarks_py(frame['right'])
            right_smooth = kalman_right.update(right_norm)
        else:
            right_smooth = np.zeros((21, 3), dtype=np.float32)
        
        # Concatenate both hands: (21*3 + 21*3) = 126
        frame_vector = np.concatenate([
            left_smooth.flatten(),
            right_smooth.flatten()
        ])
        processed_frames.append(frame_vector)
    
    # Stack to (T, 126)
    sequence = np.array(processed_frames, dtype=np.float32)
    
    # Pad or truncate to fixed sequence length
    if len(sequence) < sequence_length:
        padding = np.zeros((sequence_length - len(sequence), 126), dtype=np.float32)
        sequence = np.concatenate([sequence, padding], axis=0)
    elif len(sequence) > sequence_length:
        sequence = sequence[:sequence_length]
    
    return sequence


def preprocess_dataset(input_dir: str, output_dir: str, 
                       sequence_length: int = 64, fps: int = 30):
    """
    Preprocess an entire dataset directory, searching recursively for videos.
    The immediate parent folder of a video is used as its class label.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Added uppercase extensions to catch .MOV files
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.MOV', '.MP4'}
    
    # 1. Find all videos recursively
    all_videos = []
    for ext in video_extensions:
        all_videos.extend(input_path.rglob(f'*{ext}'))
        
    if not all_videos:
        print(f"ERROR: No videos found recursively in {input_dir}")
        return
        
    # 2. Group videos by their specific class (the folder they are immediately inside)
    class_to_videos = {}
    for video in all_videos:
        class_name = video.parent.name  # e.g., '87. hot'
        if class_name not in class_to_videos:
            class_to_videos[class_name] = []
        class_to_videos[class_name].append(video)
        
    class_names = sorted(list(class_to_videos.keys()))
    label_map = {name: i for i, name in enumerate(class_names)}
    
    all_sequences = []
    all_labels = []
    
    # 3. Process them
    for class_name in tqdm(class_names, desc="Processing classes"):
        label = label_map[class_name]
        videos = class_to_videos[class_name]
        
        for video_file in tqdm(videos, desc=f"  {class_name}", leave=False):
            sequence = process_single_video(
                str(video_file), sequence_length, fps
            )
            if sequence is not None:
                all_sequences.append(sequence)
                all_labels.append(label)
                
    if not all_sequences:
        print("ERROR: No sequences processed. MediaPipe might be failing to extract landmarks.")
        return
        
    # Save
    sequences_array = np.array(all_sequences, dtype=np.float32)
    labels_array = np.array(all_labels, dtype=np.int32)
    
    np.save(output_path / 'sequences.npy', sequences_array)
    np.save(output_path / 'labels.npy', labels_array)
    
    with open(output_path / 'label_map.json', 'w') as f:
        json.dump(label_map, f, indent=2)
        
    print(f"\nPreprocessing complete:")
    print(f"  Sequences: {sequences_array.shape}")
    print(f"  Labels:    {labels_array.shape}")
    print(f"  Classes:   {len(label_map)}")
    print(f"  Saved to:  {output_path}")

'''
def preprocess_dataset(input_dir: str, output_dir: str, 
                       sequence_length: int = 64, fps: int = 25):
    """
    Preprocess an entire dataset directory.
    
    Expected structure:
      input_dir/
        class_0/
          video1.mp4
          video2.mp4
        class_1/
          ...
    
    Output:
      output_dir/
        sequences.npy     — shape (N, sequence_length, 126)
        labels.npy        — shape (N,) integer labels
        label_map.json    — {class_name: int_label}
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Discover classes
    class_dirs = sorted([d for d in input_path.iterdir() if d.is_dir()])
    if not class_dirs:
        print(f"ERROR: No class directories found in {input_dir}")
        return
    
    label_map = {d.name: i for i, d in enumerate(class_dirs)}
    
    all_sequences = []
    all_labels = []
    
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
    
    for class_dir in tqdm(class_dirs, desc="Processing classes"):
        label = label_map[class_dir.name]
        videos = [f for f in class_dir.iterdir() 
                  if f.suffix.lower() in video_extensions]
        
        for video_file in tqdm(videos, desc=f"  {class_dir.name}", leave=False):
            sequence = process_single_video(
                str(video_file), sequence_length, fps
            )
            if sequence is not None:
                all_sequences.append(sequence)
                all_labels.append(label)
    
    if not all_sequences:
        print("ERROR: No sequences processed. Check input directory.")
        return
    
    # Save
    sequences_array = np.array(all_sequences, dtype=np.float32)
    labels_array = np.array(all_labels, dtype=np.int32)
    
    np.save(output_path / 'sequences.npy', sequences_array)
    np.save(output_path / 'labels.npy', labels_array)
    
    with open(output_path / 'label_map.json', 'w') as f:
        json.dump(label_map, f, indent=2)
    
    print(f"\nPreprocessing complete:")
    print(f"  Sequences: {sequences_array.shape}")
    print(f"  Labels:    {labels_array.shape}")
    print(f"  Classes:   {len(label_map)}")
    print(f"  Saved to:  {output_path}")

'''
# ============================================================
# Synthetic Dataset Generator (for pipeline validation)
# ============================================================

def generate_synthetic_dataset(output_dir: str, n_classes: int = 100,
                               samples_per_class: int = 20,
                               sequence_length: int = 64):
    """
    Generate a synthetic landmark dataset for pipeline validation.
    Each class gets a unique "signature" pattern in the landmark space.
    
    This is used to validate the full training pipeline before
    real data (WLASL100, INCLUDE) is available.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    np.random.seed(42)
    
    # Generate class prototypes
    class_prototypes = []
    for c in range(n_classes):
        # Each class has a distinct temporal pattern
        prototype = np.random.randn(sequence_length, 126).astype(np.float32) * 0.5
        # Add class-specific temporal dynamics
        freq = 0.1 + (c / n_classes) * 0.5
        phase = c * 2 * np.pi / n_classes
        for t in range(sequence_length):
            modulation = np.sin(freq * t + phase) * 0.3
            prototype[t, :63] += modulation  # left hand
            prototype[t, 63:] -= modulation  # right hand (inverted)
        class_prototypes.append(prototype)
    
    all_sequences = []
    all_labels = []
    
    for c in range(n_classes):
        for s in range(samples_per_class):
            # Add noise to prototype
            noise = np.random.randn(sequence_length, 126).astype(np.float32) * 0.1
            sequence = class_prototypes[c] + noise
            all_sequences.append(sequence)
            all_labels.append(c)
    
    sequences_array = np.array(all_sequences, dtype=np.float32)
    labels_array = np.array(all_labels, dtype=np.int32)
    
    # Shuffle
    idx = np.random.permutation(len(all_sequences))
    sequences_array = sequences_array[idx]
    labels_array = labels_array[idx]
    
    # Split: 80/10/10
    n = len(sequences_array)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    
    splits = {
        'train': (sequences_array[:n_train], labels_array[:n_train]),
        'val': (sequences_array[n_train:n_train+n_val], labels_array[n_train:n_train+n_val]),
        'test': (sequences_array[n_train+n_val:], labels_array[n_train+n_val:]),
    }
    
    label_map = {f"sign_{i:03d}": i for i in range(n_classes)}
    
    for split_name, (seqs, labs) in splits.items():
        split_dir = output_path / split_name
        split_dir.mkdir(exist_ok=True)
        np.save(split_dir / 'sequences.npy', seqs)
        np.save(split_dir / 'labels.npy', labs)
    
    with open(output_path / 'label_map.json', 'w') as f:
        json.dump(label_map, f, indent=2)
    
    # Also save a small representative dataset for quantization
    rep_indices = np.random.choice(n_train, min(200, n_train), replace=False)
    rep_dataset = sequences_array[rep_indices]
    np.save(output_path / 'rep_dataset.npy', rep_dataset)
    
    print(f"Synthetic dataset generated:")
    print(f"  Classes:     {n_classes}")
    print(f"  Train:       {splits['train'][0].shape}")
    print(f"  Validation:  {splits['val'][0].shape}")
    print(f"  Test:        {splits['test'][0].shape}")
    print(f"  Rep dataset: {rep_dataset.shape}")
    print(f"  Saved to:    {output_path}")


# ============================================================
# CLI
# ============================================================

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='SignBridge Landmark Preprocessing')
    subparsers = parser.add_subparsers(dest='command')
    
    # Real dataset preprocessing
    proc = subparsers.add_parser('process', help='Process video dataset')
    proc.add_argument('--input_dir', required=True, help='Input video directory')
    proc.add_argument('--output_dir', required=True, help='Output .npy directory')
    proc.add_argument('--sequence_length', type=int, default=64)
    proc.add_argument('--fps', type=int, default=25)
    
    # Synthetic dataset generation
    syn = subparsers.add_parser('synthetic', help='Generate synthetic dataset')
    syn.add_argument('--output_dir', required=True, help='Output directory')
    syn.add_argument('--n_classes', type=int, default=100)
    syn.add_argument('--samples_per_class', type=int, default=20)
    syn.add_argument('--sequence_length', type=int, default=64)
    
    args = parser.parse_args()
    
    if args.command == 'process':
        preprocess_dataset(args.input_dir, args.output_dir,
                          args.sequence_length, args.fps)
    elif args.command == 'synthetic':
        generate_synthetic_dataset(args.output_dir, args.n_classes,
                                  args.samples_per_class, args.sequence_length)
    else:
        parser.print_help()


In [ ]:
%%writefile train.py
"""
train.py
SignBridge ML — Model Training Script

Usage:
  python train.py \
    --dataset data/processed \
    --epochs 100 \
    --batch_size 32 \
    --learning_rate 0.001 \
    --early_stopping_patience 10 \
    --output models/tms_wlasl100.h5
"""

import argparse
import os
import sys
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

# Add parent src to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))
from tms_model import build_tms_model


def load_dataset(data_dir: str):
    """
    Load preprocessed dataset from .npy files.
    
    Expected structure:
      data_dir/
        train/sequences.npy, labels.npy
        val/sequences.npy, labels.npy
        test/sequences.npy, labels.npy
        label_map.json
    """
    data_path = Path(data_dir)
    
    # Load label map
    with open(data_path / 'label_map.json', 'r') as f:
        label_map = json.load(f)
    n_classes = len(label_map)
    
    splits = {}
    for split in ['train', 'val', 'test']:
        split_dir = data_path / split
        sequences = np.load(split_dir / 'sequences.npy')
        labels = np.load(split_dir / 'labels.npy')
        
        # One-hot encode labels
        labels_onehot = keras.utils.to_categorical(labels, num_classes=n_classes)
        
        splits[split] = (sequences, labels_onehot)
        print(f"  {split}: sequences={sequences.shape}, labels={labels_onehot.shape}")
    
    return splits, label_map, n_classes


def augment_data(sequences: np.ndarray, labels: np.ndarray,
                 rotation_range: float = 15.0, time_warp: bool = True):
    """
    Data augmentation for landmark sequences.
    
    - Random rotation ±15° around Z-axis (applied to each hand's landmarks)
    - Time warping: randomly stretch/compress temporal dimension
    - Random noise injection
    """
    augmented_seqs = []
    augmented_labels = []
    
    for seq, label in zip(sequences, labels):
        # Original
        augmented_seqs.append(seq)
        augmented_labels.append(label)
        
        # Augmented copy 1: rotation
        aug1 = seq.copy()
        angle = np.random.uniform(-rotation_range, rotation_range) * np.pi / 180
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        
        for t in range(len(aug1)):
            # Rotate left hand (indices 0-62, every 3 values = x,y,z)
            for i in range(0, 63, 3):
                x, y = aug1[t, i], aug1[t, i+1]
                aug1[t, i] = x * cos_a - y * sin_a
                aug1[t, i+1] = x * sin_a + y * cos_a
            # Rotate right hand (indices 63-125)
            for i in range(63, 126, 3):
                x, y = aug1[t, i], aug1[t, i+1]
                aug1[t, i] = x * cos_a - y * sin_a
                aug1[t, i+1] = x * sin_a + y * cos_a
        
        augmented_seqs.append(aug1)
        augmented_labels.append(label)
        
        # Augmented copy 2: noise
        aug2 = seq + np.random.randn(*seq.shape).astype(np.float32) * 0.02
        augmented_seqs.append(aug2)
        augmented_labels.append(label)
    
    return np.array(augmented_seqs), np.array(augmented_labels)


def train(args):
    """Main training loop."""
    print("="*60)
    print("  SignBridge TMS-Attention Model Training")
    print("="*60)
    
    # Load data
    print(f"\nLoading dataset from: {args.dataset}")
    splits, label_map, n_classes = load_dataset(args.dataset)
    
    X_train, y_train = splits['train']
    X_val, y_val = splits['val']
    X_test, y_test = splits['test']
    
    seq_len = X_train.shape[1]
    n_features = X_train.shape[2]
    
    print(f"\nDataset summary:")
    print(f"  Sequence length: {seq_len}")
    print(f"  Features:        {n_features}")
    print(f"  Classes:         {n_classes}")
    
    # Augment training data
    if args.augment:
        print("\nApplying data augmentation...")
        X_train, y_train = augment_data(X_train, y_train)
        print(f"  After augmentation: {X_train.shape}")
    
    # Build model
    print("\nBuilding TMS-Attention model...")
    model = build_tms_model(
        sequence_length=seq_len,
        n_features=n_features,
        n_classes=n_classes,
        d_model=128,
        num_heads=4,
        dropout_rate=0.3,
        label_smoothing=args.label_smoothing
    )
    
    # Update learning rate if specified
    if args.learning_rate != 0.001:
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=args.label_smoothing),
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
        )
    
    try:
        model.summary(line_length=120)
    except ValueError:
        print("  (model.summary skipped — console too narrow)")
    print(f"\nTotal parameters: {model.count_params():,}")
    
    # Callbacks
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=args.early_stopping_patience,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            str(output_path),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
    ]
    
    # Train
    print(f"\nTraining for up to {args.epochs} epochs...")
    print(f"  Batch size: {args.batch_size}")
    print(f"  Learning rate: {args.learning_rate}")
    print(f"  Early stopping patience: {args.early_stopping_patience}")
    print(f"  Label smoothing: {args.label_smoothing}")
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=callbacks,
        verbose=1
    )
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_results = model.evaluate(X_test, y_test, verbose=0)
    test_loss, test_acc, test_top5 = test_results
    
    print(f"\n{'='*60}")
    print(f"  Training Complete")
    print(f"{'='*60}")
    print(f"  Best val accuracy: {max(history.history['val_accuracy']):.4f}")
    print(f"  Test accuracy:     {test_acc:.4f}")
    print(f"  Test top-5 acc:    {test_top5:.4f}")
    print(f"  Test loss:         {test_loss:.4f}")
    print(f"  Model saved to:    {output_path}")
    
    # Save training history
    history_path = output_path.parent / f"{output_path.stem}_history.json"
    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(history_path, 'w') as f:
        json.dump(history_data, f, indent=2)
    print(f"  History saved to:  {history_path}")
    
    return model, history


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='SignBridge TMS Model Training')
    parser.add_argument('--dataset', required=True, help='Path to preprocessed dataset')
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--learning_rate', type=float, default=0.001)
    parser.add_argument('--early_stopping_patience', type=int, default=10)
    parser.add_argument('--label_smoothing', type=float, default=0.1)
    parser.add_argument('--output', default='models/tms_wlasl100.h5')
    parser.add_argument('--augment', action='store_true', default=True)
    parser.add_argument('--no-augment', dest='augment', action='store_false')
    
    args = parser.parse_args()
    train(args)


In [ ]:
%%writefile evaluate.py

"""
evaluate.py
SignBridge ML — Model Evaluation & CHECK-2 Validation

Measures all CHECK-2 metrics against required thresholds:
  1. Top-1 accuracy   ≥ 90 %
  2. Top-5 accuracy   ≥ 97 %
  3. Mean inference    ≤ 65 ms
  4. Model size (int8) ≤ 8 MB
  5. Macro-F1 score    ≥ 0.85
  6. False Acceptance  ≤ 5 %
  7. Peak memory       ≤ 80 MB

Usage:
  python evaluate.py \
    --model models/tms_wlasl100_int8.tflite \
    --dataset data/processed \
    --report models/check2_report.json
"""

import argparse
import json
import os
import sys
import time
import tracemalloc
import numpy as np
from pathlib import Path


# ====================================================================
# CHECK-2 Thresholds (from master execution document Table 6)
# ====================================================================
THRESHOLDS = {
    'top1_accuracy':      0.90,
    'top5_accuracy':      0.97,
    'mean_inference_ms':  65.0,
    'model_size_mb':      8.0,
    'macro_f1':           0.85,
    'false_acceptance':   0.05,
    'peak_memory_mb':     80.0,
}


# ====================================================================
# Metric helpers
# ====================================================================

def top_k_accuracy(y_true: np.ndarray, y_pred: np.ndarray, k: int) -> float:
    """Compute top-k accuracy from integer labels and probabilities."""
    top_k_preds = np.argsort(y_pred, axis=1)[:, -k:]
    correct = np.array([y_true[i] in top_k_preds[i] for i in range(len(y_true))])
    return float(correct.mean())


def macro_f1_score(y_true: np.ndarray, y_pred_classes: np.ndarray,
                   n_classes: int) -> float:
    """Compute macro-averaged F1 without sklearn dependency."""
    f1_scores = []
    for c in range(n_classes):
        tp = int(np.sum((y_pred_classes == c) & (y_true == c)))
        fp = int(np.sum((y_pred_classes == c) & (y_true != c)))
        fn = int(np.sum((y_pred_classes != c) & (y_true == c)))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) \
            if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)

    return float(np.mean(f1_scores))


def false_acceptance_rate(y_true: np.ndarray, y_pred_probs: np.ndarray,
                          threshold: float = 0.5) -> float:
    """
    False Acceptance Rate — fraction of *incorrect* predictions whose
    confidence exceeds `threshold`.  Measures overconfident errors.
    """
    pred_classes = np.argmax(y_pred_probs, axis=1)
    max_conf = np.max(y_pred_probs, axis=1)
    incorrect_mask = pred_classes != y_true
    n_incorrect = int(np.sum(incorrect_mask))
    if n_incorrect == 0:
        return 0.0
    false_accepts = int(np.sum(max_conf[incorrect_mask] >= threshold))
    return false_accepts / len(y_true)


# ====================================================================
# Main evaluation
# ====================================================================

def evaluate(model_path: str, dataset_dir: str,
             report_path: str | None = None):
    """
    Run full CHECK-2 evaluation on a TFLite model.

    Returns:
        dict with all metrics and pass/fail status.
    """
    import tensorflow as tf

    dataset_path = Path(dataset_dir)

    # ---- Load test data ----
    test_dir = dataset_path / 'test'
    X_test = np.load(test_dir / 'sequences.npy').astype(np.float32)
    y_test = np.load(test_dir / 'labels.npy').astype(np.int32)

    with open(dataset_path / 'label_map.json', 'r') as f:
        label_map = json.load(f)
    n_classes = len(label_map)

    print(f"Test set : {X_test.shape[0]} samples, {n_classes} classes")
    print(f"Model    : {model_path}")
    print()

    # ---- Model size ----
    model_size_bytes = os.path.getsize(model_path)
    model_size_mb = model_size_bytes / (1024 * 1024)

    # ---- Load interpreter ----
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # ---- Inference + timing + memory ----
    tracemalloc.start()
    all_probs = []
    latencies = []

    for i in range(len(X_test)):
        sample = X_test[i:i+1]
        interpreter.set_tensor(input_details[0]['index'], sample)

        t0 = time.perf_counter()
        interpreter.invoke()
        t1 = time.perf_counter()

        latencies.append((t1 - t0) * 1000.0)  # ms
        probs = interpreter.get_tensor(output_details[0]['index'])[0]
        all_probs.append(probs)

    _, peak_memory_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_memory_mb = peak_memory_bytes / (1024 * 1024)

    all_probs = np.array(all_probs)
    pred_classes = np.argmax(all_probs, axis=1)

    # ---- Compute metrics ----
    metrics = {
        'top1_accuracy':     top_k_accuracy(y_test, all_probs, k=1),
        'top5_accuracy':     top_k_accuracy(y_test, all_probs, k=5),
        'mean_inference_ms': float(np.mean(latencies)),
        'p95_inference_ms':  float(np.percentile(latencies, 95)),
        'model_size_mb':     round(model_size_mb, 3),
        'macro_f1':          macro_f1_score(y_test, pred_classes, n_classes),
        'false_acceptance':  false_acceptance_rate(y_test, all_probs),
        'peak_memory_mb':    round(peak_memory_mb, 2),
        'n_samples':         int(len(y_test)),
        'n_classes':         n_classes,
    }

    # ---- Pass / Fail ----
    checks = {}
    checks['top1_accuracy']     = metrics['top1_accuracy']     >= THRESHOLDS['top1_accuracy']
    checks['top5_accuracy']     = metrics['top5_accuracy']     >= THRESHOLDS['top5_accuracy']
    checks['mean_inference_ms'] = metrics['mean_inference_ms'] <= THRESHOLDS['mean_inference_ms']
    checks['model_size_mb']     = metrics['model_size_mb']     <= THRESHOLDS['model_size_mb']
    checks['macro_f1']          = metrics['macro_f1']          >= THRESHOLDS['macro_f1']
    checks['false_acceptance']  = metrics['false_acceptance']  <= THRESHOLDS['false_acceptance']
    checks['peak_memory_mb']    = metrics['peak_memory_mb']    <= THRESHOLDS['peak_memory_mb']

    all_pass = all(checks.values())

    # ---- Pretty print ----
    print("=" * 62)
    print("  CHECK 2 — Model Validation Results")
    print("=" * 62)
    fmt = "  {:<22} {:>10}  threshold {:<10}  {}"
    print(fmt.format("METRIC", "VALUE", "", "STATUS"))
    print("-" * 62)

    def fmtval(key):
        v = metrics[key]
        t = THRESHOLDS[key]
        ok = checks[key]
        if 'accuracy' in key or 'f1' in key or 'acceptance' in key:
            vstr = f"{v*100:.2f}%"
            tstr = f"{'≥' if 'acceptance' not in key else '≤'}{t*100:.0f}%"
        elif 'ms' in key:
            vstr = f"{v:.1f} ms"
            tstr = f"≤{t:.0f} ms"
        elif 'mb' in key.lower():
            vstr = f"{v:.2f} MB"
            tstr = f"≤{t:.0f} MB"
        else:
            vstr = str(v)
            tstr = str(t)
        status = "PASS" if ok else "** FAIL **"
        return vstr, tstr, status

    for key in THRESHOLDS:
        vstr, tstr, status = fmtval(key)
        print(fmt.format(key, vstr, tstr, status))

    print("-" * 62)
    overall = "ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"
    print(f"  Overall: {overall}")
    print("=" * 62)

    # ---- Per-class breakdown (top 10 worst) ----
    print("\n  Per-class accuracy (10 worst):")
    per_class_acc = []
    for c in range(n_classes):
        mask = y_test == c
        if mask.sum() == 0:
            continue
        acc = float((pred_classes[mask] == c).mean())
        per_class_acc.append((label_map.get(str(c), str(c)), acc, int(mask.sum())))

    per_class_acc.sort(key=lambda x: x[1])
    for name, acc, count in per_class_acc[:10]:
        print(f"    {name:<20} {acc*100:6.2f}%  (n={count})")

    # ---- Save report ----
    report = {
        'model': model_path,
        'dataset': dataset_dir,
        'metrics': metrics,
        'thresholds': {k: v for k, v in THRESHOLDS.items()},
        'checks': checks,
        'all_pass': all_pass,
    }

    if report_path:
        rp = Path(report_path)
        rp.parent.mkdir(parents=True, exist_ok=True)
        with open(rp, 'w') as f:
            json.dump(report, f, indent=2)
        print(f"\n  Report saved to: {report_path}")

    return report


# ====================================================================
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='SignBridge CHECK-2 Evaluation')
    parser.add_argument('--model', required=True,
                        help='Path to .tflite model')
    parser.add_argument('--dataset', required=True,
                        help='Path to preprocessed dataset directory')
    parser.add_argument('--report', default=None,
                        help='Path for JSON report output')
    args = parser.parse_args()

    report = evaluate(args.model, args.dataset, args.report)

    # Exit code: 0 if all pass, 1 if any fail
    sys.exit(0 if report['all_pass'] else 1)


In [ ]:
%%writefile convert_tflite.py

"""
convert_tflite.py
SignBridge ML — TFLite Conversion with Quantization

Usage:
  python convert_tflite.py \
    --model models/tms_wlasl100.h5 \
    --output models/tms_wlasl100_int8.tflite \
    --quantize int8 \
    --representative_dataset data/processed/rep_dataset.npy
"""

import argparse
import os
import sys
import numpy as np
import tensorflow as tf
from pathlib import Path

# Add parent src to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))

# Import custom layers so they are registered before model load
import tms_model  # noqa: F401 — registers @register_keras_serializable layers


def convert_to_tflite(model_path: str, output_path: str,
                      quantize: str = 'none',
                      representative_dataset_path: str = None):
    """
    Convert a Keras .h5 model to TFLite format.
    
    Args:
        model_path: Path to .h5 model file
        output_path: Path for output .tflite file
        quantize: Quantization mode: 'none', 'float16', 'int8'
        representative_dataset_path: Path to .npy representative dataset (for int8)
    """
    print(f"Loading model from: {model_path}")
    model = tf.keras.models.load_model(model_path, compile=False)
    
    print(f"Model loaded. Parameters: {model.count_params():,}")
    
    # Convert to TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantize == 'float16':
        print("Applying float16 quantization...")
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    
    elif quantize == 'int8':
        print("Applying int8 quantization...")
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        
        if representative_dataset_path:
            print(f"Loading representative dataset: {representative_dataset_path}")
            rep_data = np.load(representative_dataset_path).astype(np.float32)
            
            def representative_dataset_gen():
                for i in range(min(len(rep_data), 200)):
                    yield [rep_data[i:i+1]]
            
            converter.representative_dataset = representative_dataset_gen
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.float32   # Keep float input for ease of use
            converter.inference_output_type = tf.float32  # Keep float output
        else:
            print("WARNING: No representative dataset provided. Using dynamic range quantization.")
    
    elif quantize == 'none':
        print("No quantization applied (float32).")
    
    else:
        raise ValueError(f"Unknown quantization mode: {quantize}")
    
    print("Converting...")
    tflite_model = converter.convert()
    
    # Save
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    # Report size
    size_bytes = os.path.getsize(output_path)
    size_mb = size_bytes / (1024 * 1024)
    
    print(f"\nConversion complete:")
    print(f"  Output:       {output_path}")
    print(f"  Size:         {size_mb:.2f} MB ({size_bytes:,} bytes)")
    print(f"  Quantization: {quantize}")
    
    # Verify the converted model works
    print("\nVerifying converted model...")
    interpreter = tf.lite.Interpreter(model_path=output_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"  Input:  {input_details[0]['shape']} dtype={input_details[0]['dtype']}")
    print(f"  Output: {output_details[0]['shape']} dtype={output_details[0]['dtype']}")
    
    # Run a test inference
    test_input = np.random.randn(*input_details[0]['shape']).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()
    test_output = interpreter.get_tensor(output_details[0]['index'])
    
    print(f"  Test output shape: {test_output.shape}")
    print(f"  Test output sum:   {test_output.sum():.4f} (expected ~1.0)")
    print(f"  Verification: PASSED")
    
    return output_path, size_mb


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='SignBridge TFLite Conversion')
    parser.add_argument('--model', required=True, help='Path to .h5 model')
    parser.add_argument('--output', required=True, help='Path for .tflite output')
    parser.add_argument('--quantize', choices=['none', 'float16', 'int8'], default='int8')
    parser.add_argument('--representative_dataset', default=None,
                        help='Path to .npy representative dataset (for int8 quantization)')
    
    args = parser.parse_args()
    convert_to_tflite(args.model, args.output, args.quantize,
                      args.representative_dataset)


In [ ]:
%%writefile tms_model.py

"""
tms_model.py
SignBridge ML — TMS-Attention Model Architecture

Architecture:
  Input: (batch, 64, 126) — sequence of normalized landmark vectors

  Branch A (local spatial): 1D Conv layers (MobileNet-style depthwise separable)
    Kernel sizes: 3, 5, 7. Output: (batch, 64, 128)

  Branch B (temporal attention): Multi-Head Self-Attention (4 heads, d_model=128)
    Positional encoding added. Output: (batch, 64, 128)

  Conformer block between Conv output and Attention
    Conv module: Pointwise → Depthwise Conv → GLU → Pointwise
    Self-attention module: Multi-head attention with relative positional encoding
    Feed-forward module: Two linear layers with SiLU activation

  Concatenate: (batch, 64, 256) → Global Average Pooling → Dense(128, ReLU)
    → Dropout(0.3) → Dense(N_CLASSES, Softmax)

  CTC decode layer: wraps model in CTC framework for continuous signing.
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import numpy as np


# ============================================================
# Positional Encoding
# ============================================================

@keras.utils.register_keras_serializable(package='SignBridge')
class PositionalEncoding(layers.Layer):
    """Sinusoidal positional encoding for transformer-style attention."""
    
    def __init__(self, max_len=128, d_model=128, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model
        
        # Precompute positional encoding matrix
        pe = np.zeros((max_len, d_model), dtype=np.float32)
        position = np.arange(0, max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        
        self.pe = tf.constant(pe[np.newaxis, :, :])  # (1, max_len, d_model)
    
    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]
    
    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len, 'd_model': self.d_model})
        return config


# ============================================================
# Branch A: MobileNet-style Depthwise Separable Conv
# ============================================================

@keras.utils.register_keras_serializable(package='SignBridge')
class DepthwiseSeparableConv1D(layers.Layer):
    """MobileNet-style depthwise separable 1D convolution."""
    
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self._filters = filters
        self._kernel_size = kernel_size
        self.depthwise = layers.DepthwiseConv1D(
            kernel_size=kernel_size, padding='same', activation=None
        )
        self.pointwise = layers.Conv1D(filters, 1, activation=None)
        self.bn = layers.BatchNormalization()
        self.activation = layers.ReLU()
    
    def call(self, x, training=False):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x, training=training)
        x = self.activation(x)
        return x
    
    def get_config(self):
        config = super().get_config()
        config.update({'filters': self._filters, 'kernel_size': self._kernel_size})
        return config


def build_conv_branch(input_shape, filters=128):
    """
    Branch A: Multi-scale 1D depthwise separable convolutions.
    Kernel sizes: 3, 5, 7 — captures local spatial patterns at different scales.
    
    Input:  (batch, seq_len, features)
    Output: (batch, seq_len, filters)
    """
    inputs = layers.Input(shape=input_shape)
    
    # Project to working dimension
    x = layers.Dense(filters)(inputs)
    
    # Multi-scale convolutions
    conv3 = DepthwiseSeparableConv1D(filters // 3, kernel_size=3)(x)
    conv5 = DepthwiseSeparableConv1D(filters // 3, kernel_size=5)(x)
    conv7 = DepthwiseSeparableConv1D(filters - 2 * (filters // 3), kernel_size=7)(x)
    
    # Concatenate multi-scale features
    x = layers.Concatenate()([conv3, conv5, conv7])
    
    # Additional refinement
    x = DepthwiseSeparableConv1D(filters, kernel_size=3)(x)
    
    return Model(inputs, x, name='conv_branch')


# ============================================================
# Conformer Block (Step 2.3)
# ============================================================

@keras.utils.register_keras_serializable(package='SignBridge')
class ConformerFeedForward(layers.Layer):
    """Feed-forward module: two linear layers with SiLU activation."""
    
    def __init__(self, d_model=128, expansion_factor=4, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self._d_model = d_model
        self._expansion_factor = expansion_factor
        self._dropout_rate = dropout_rate
        self.ln = layers.LayerNormalization()
        self.dense1 = layers.Dense(d_model * expansion_factor)
        self.dense2 = layers.Dense(d_model)
        self.dropout = layers.Dropout(dropout_rate)
    
    def call(self, x, training=False):
        residual = x
        x = self.ln(x)
        x = self.dense1(x)
        x = tf.nn.silu(x)  # SiLU activation
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        x = self.dropout(x, training=training)
        return residual + 0.5 * x
    
    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self._d_model, 'expansion_factor': self._expansion_factor, 'dropout_rate': self._dropout_rate})
        return config


@keras.utils.register_keras_serializable(package='SignBridge')
class ConformerConvModule(layers.Layer):
    """
    Conformer conv module:
    Pointwise → Depthwise Conv → GLU → Pointwise
    """
    
    def __init__(self, d_model=128, kernel_size=31, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self._d_model = d_model
        self._kernel_size = kernel_size
        self._dropout_rate = dropout_rate
        self.ln = layers.LayerNormalization()
        self.pointwise1 = layers.Dense(2 * d_model)
        self.depthwise = layers.DepthwiseConv1D(
            kernel_size=kernel_size, padding='same'
        )
        self.bn = layers.BatchNormalization()
        self.pointwise2 = layers.Dense(d_model)
        self.dropout = layers.Dropout(dropout_rate)
    
    def call(self, x, training=False):
        residual = x
        x = self.ln(x)
        x = self.pointwise1(x)
        
        # GLU (Gated Linear Unit)
        x1, x2 = tf.split(x, 2, axis=-1)
        x = x1 * tf.nn.sigmoid(x2)
        
        x = self.depthwise(x)
        x = self.bn(x, training=training)
        x = tf.nn.silu(x)
        x = self.pointwise2(x)
        x = self.dropout(x, training=training)
        
        return residual + x
    
    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self._d_model, 'kernel_size': self._kernel_size, 'dropout_rate': self._dropout_rate})
        return config


@keras.utils.register_keras_serializable(package='SignBridge')
class ConformerBlock(layers.Layer):
    """
    Full Conformer block:
    FF → MHSA → Conv → FF
    
    Replaces LSTM for better generalization with full batch parallelism.
    """
    
    def __init__(self, d_model=128, num_heads=4, conv_kernel_size=31,
                 ff_expansion=4, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self._d_model = d_model
        self._num_heads = num_heads
        self._conv_kernel_size = conv_kernel_size
        self._ff_expansion = ff_expansion
        self._dropout_rate = dropout_rate
        self.ff1 = ConformerFeedForward(d_model, ff_expansion, dropout_rate)
        self.mhsa_ln = layers.LayerNormalization()
        self.mhsa = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.mhsa_dropout = layers.Dropout(dropout_rate)
        self.conv_module = ConformerConvModule(d_model, conv_kernel_size, dropout_rate)
        self.ff2 = ConformerFeedForward(d_model, ff_expansion, dropout_rate)
        self.final_ln = layers.LayerNormalization()
    
    def call(self, x, training=False):
        # Feed-forward 1
        x = self.ff1(x, training=training)
        
        # Multi-head self-attention
        residual = x
        x_ln = self.mhsa_ln(x)
        attn_out = self.mhsa(x_ln, x_ln, training=training)
        x = residual + self.mhsa_dropout(attn_out, training=training)
        
        # Convolution module
        x = self.conv_module(x, training=training)
        
        # Feed-forward 2
        x = self.ff2(x, training=training)
        
        # Final layer norm
        x = self.final_ln(x)
        
        return x
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self._d_model,
            'num_heads': self._num_heads,
            'conv_kernel_size': self._conv_kernel_size,
            'ff_expansion': self._ff_expansion,
            'dropout_rate': self._dropout_rate,
        })
        return config


# ============================================================
# Branch B: Multi-Head Self-Attention with Positional Encoding
# ============================================================

def build_attention_branch(input_shape, d_model=128, num_heads=4):
    """
    Branch B: Temporal attention (transformer-style).
    
    Input:  (batch, seq_len, features)
    Output: (batch, seq_len, d_model)
    """
    inputs = layers.Input(shape=input_shape)
    
    # Project to d_model
    x = layers.Dense(d_model)(inputs)
    
    # Add positional encoding
    x = PositionalEncoding(max_len=input_shape[0], d_model=d_model)(x)
    
    # Multi-Head Self-Attention
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=0.1
    )(x, x)
    x = layers.Add()([x, attn_output])
    x = layers.LayerNormalization()(x)
    
    return Model(inputs, x, name='attention_branch')


# ============================================================
# Full TMS-Attention Model
# ============================================================

def build_tms_model(sequence_length: int = 64, n_features: int = 126,
                    n_classes: int = 100, d_model: int = 128,
                    num_heads: int = 4, dropout_rate: float = 0.3,
                    label_smoothing: float = 0.1):
    """
    Build the full TMS-Attention model with Conformer block.
    
    Args:
        sequence_length: Number of frames in input sequence (default 64)
        n_features: Number of features per frame (126 = 2 hands × 21 × 3)
        n_classes: Number of sign classes
        d_model: Model dimension
        num_heads: Number of attention heads
        dropout_rate: Dropout rate for classification head
        label_smoothing: Label smoothing factor
    
    Returns:
        Compiled Keras model
    """
    inputs = layers.Input(shape=(sequence_length, n_features), name='landmarks')
    
    # ---- Branch A: Depthwise Separable Convolutions ----
    conv_branch = build_conv_branch((sequence_length, n_features), d_model)
    conv_out = conv_branch(inputs)  # (batch, 64, 128)
    
    # ---- Conformer Block (between Conv and Attention) ----
    conformer = ConformerBlock(
        d_model=d_model, num_heads=num_heads,
        conv_kernel_size=15,  # smaller kernel for short sequences
        dropout_rate=0.1
    )
    conv_out = conformer(conv_out)  # (batch, 64, 128)
    
    # ---- Branch B: Multi-Head Self-Attention ----
    attn_branch = build_attention_branch((sequence_length, n_features), d_model, num_heads)
    attn_out = attn_branch(inputs)  # (batch, 64, 128)
    
    # ---- Concatenate branches ----
    merged = layers.Concatenate()([conv_out, attn_out])  # (batch, 64, 256)
    
    # ---- Classification Head ----
    x = layers.GlobalAveragePooling1D()(merged)  # (batch, 256)
    x = layers.Dense(d_model, activation='relu')(x)  # (batch, 128)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='classification')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='TMS_Attention')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
    )
    
    return model


# ============================================================
# CTC Model Wrapper (for continuous signing)
# ============================================================

def build_ctc_model(sequence_length: int = 64, n_features: int = 126,
                    n_classes: int = 100, d_model: int = 128):
    """
    Build CTC-wrapped model for continuous sign recognition.
    
    The base model outputs per-frame class probabilities,
    and CTC loss handles alignment between input frames and label sequence.
    """
    inputs = layers.Input(shape=(sequence_length, n_features), name='landmarks')
    
    # Shared feature extraction
    conv_branch = build_conv_branch((sequence_length, n_features), d_model)
    conv_out = conv_branch(inputs)
    
    conformer = ConformerBlock(d_model=d_model, num_heads=4, conv_kernel_size=15)
    conv_out = conformer(conv_out)
    
    attn_branch = build_attention_branch((sequence_length, n_features), d_model, 4)
    attn_out = attn_branch(inputs)
    
    merged = layers.Concatenate()([conv_out, attn_out])  # (batch, 64, 256)
    
    # Per-frame classification (for CTC)
    x = layers.Dense(d_model, activation='relu')(merged)
    x = layers.Dropout(0.3)(x)
    # +1 for CTC blank label
    outputs = layers.Dense(n_classes + 1, activation='softmax', name='ctc_output')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='TMS_Attention_CTC')
    return model


# ============================================================
# Model Summary (for testing)
# ============================================================

if __name__ == '__main__':
    print("Building TMS-Attention model (100 classes)...")
    model = build_tms_model(n_classes=100)
    model.summary()
    
    print(f"\nTotal parameters: {model.count_params():,}")
    
    # Test forward pass
    dummy_input = np.random.randn(2, 64, 126).astype(np.float32)
    output = model.predict(dummy_input, verbose=0)
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Output sums:  {output.sum(axis=1)}")  # Should be ~1.0
    
    print("\nModel architecture built successfully!")


In [ ]:
import os

dataset_path = '/kaggle/input/datasets/daskoushik/include'

# Count top-level folders
classes = [d for d in os.listdir(dataset_path) 
           if os.path.isdir(os.path.join(dataset_path, d))]
print(f'Number of top-level folders: {len(classes)}')
print(f'First 10 folders: {classes[:10]}')

# Count videos per folder recursively
video_counts = {}
for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    video_count = 0
    
    # os.walk traverses all subdirectories recursively
    for root, dirs, files in os.walk(cls_path):
        for f in files:
            # .lower() makes the check case-insensitive (.MOV becomes .mov)
            if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                video_count += 1
                
    video_counts[cls] = video_count

total = sum(video_counts.values())
avg = total / len(classes)
mn = min(video_counts.values())
mx = max(video_counts.values())

print(f'\nTotal videos: {total}')
print(f'Average per top folder: {avg:.1f}')
print(f'Minimum per top folder: {mn}')
print(f'Maximum per top folder: {mx}')

In [ ]:
import os
import cv2

dataset_path = '/kaggle/input/datasets/daskoushik/include'

# 1. Safely find the first available video file
sample_video = None
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        # Check for multiple extensions, ignoring case
        if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            sample_video = os.path.join(root, f)
            break  # Stop checking files in this folder once we find one
    if sample_video:
        break  # Stop walking folders completely once we have a video

# 2. Inspect the video properties using OpenCV
if sample_video:
    cap = cv2.VideoCapture(sample_video)
    
    # Check if OpenCV successfully opened the file
    if not cap.isOpened():
        print(f"Error: OpenCV could not open the video {sample_video}")
    else:
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        
        # Calculate duration (handling potential division by zero just in case)
        duration = frames / fps if fps > 0 else 0
        
        print(f'Sample video: {sample_video}')
        print(f'FPS: {fps}')
        print(f'Frames: {frames}')
        print(f'Duration: {duration:.2f} seconds')
        print(f'Resolution: {w}x{h}')
else:
    print("Could not find any video files in the specified dataset path.")

In [ ]:
import urllib.request, os
os.makedirs('/kaggle/working/models', exist_ok=True)
url = ('https://storage.googleapis.com/mediapipe-models/'
 
'hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task')
print('Downloading hand_landmarker.task...')
urllib.request.urlretrieve(url, 
'/kaggle/working/models/hand_landmarker.task')
size = os.path.getsize('/kaggle/working/models/hand_landmarker.task')
print(f'Downloaded: {size/1024/1024:.1f} MB')
# Expected: ~29 MB

In [ ]:
# 1. Install MediaPipe
!pip install -q mediapipe

# 2. Download the MediaPipe Hand Landmarker model file
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task -O /kaggle/working/hand_landmarker.task

In [ ]:
import os

# Create output directory
os.makedirs('/kaggle/working/data/processed', exist_ok=True)

# Run preprocess.py
!python /kaggle/working/preprocess.py process \
 --input_dir /kaggle/input/datasets/daskoushik/include \
 --output_dir /kaggle/working/data/processed \
 --sequence_length 64 \
 --fps 25

In [ ]:
#Preprocess verification

import numpy as np
import json

base = '/kaggle/working/data/processed'

# Load the combined data directly from the base directory
seq = np.load(f'{base}/sequences.npy')
lbl = np.load(f'{base}/labels.npy')
lm = json.load(open(f'{base}/label_map.json'))

print('=== Preprocessing Verification ===')
print(f'Sequences shape   : {seq.shape}')      # Should be (4284, 64, 126)
print(f'Labels shape      : {lbl.shape}')      # Should be (4284,)
print(f'Number of classes : {len(lm)}')        # Should be 264
print(f'Value range       : [{seq.min():.4f}, {seq.max():.4f}]')

print(f'NaN count         : {np.isnan(seq).sum()}')  # Must be 0
print(f'Inf count         : {np.isinf(seq).sum()}')  # Must be 0
print(f'Unique label count: {len(np.unique(lbl))}')  # Should equal 264

assert np.isnan(seq).sum() == 0, 'NaN values found!'
assert np.isinf(seq).sum() == 0, 'Inf values found!'
print('\n✓ All checks passed. Ready to split and train.')

In [ ]:
import tensorflow as tf

print('TensorFlow version :', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
print('GPUs available :', gpus)

# Expected output:
# TensorFlow version : 2.x.x
# GPUs available : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

if not gpus:
    print('WARNING: No GPU detected. Go to Session Options and enable P100 or T4 GPU.')
else:
    print('GPU is ready for training.')

In [ ]:
!mkdir -p /kaggle/working/src

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

base = '/kaggle/working/data/processed'

# 1. Load the master dataset
X = np.load(f'{base}/sequences.npy')
y = np.load(f'{base}/labels.npy')

print(f"Loaded master dataset: {X.shape}")

# 2. Split: 80% Train, 20% Temp
# We keep stratify here so the Training set learns every single class
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Split Temp: 10% Val, 10% Test 
# Removed stratify here to prevent the 1-sample crash
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# 4. Save to subfolders
splits = {
    'train': (X_train, y_train),
    'val': (X_val, y_val),
    'test': (X_test, y_test)
}

for split_name, (X_data, y_data) in splits.items():
    split_dir = f'{base}/{split_name}'
    os.makedirs(split_dir, exist_ok=True)
    np.save(f'{split_dir}/sequences.npy', X_data)
    np.save(f'{split_dir}/labels.npy', y_data)

print(f"✓ Data successfully organized!")
print(f"  Train : {X_train.shape[0]} videos")
print(f"  Val   : {X_val.shape[0]} videos")
print(f"  Test  : {X_test.shape[0]} videos")

In [ ]:
!python /kaggle/working/train.py \
 --dataset /kaggle/working/data/processed \
 --epochs 100 \
 --batch_size 32 \
 --learning_rate 0.001 \
 --early_stopping_patience 10 \
 --label_smoothing 0.1 \
 --augment \
 --output /kaggle/working/models/tms_isl.h5

In [ ]:
import numpy as np
import os

base = '/kaggle/working/data/processed'
X_train = np.load(f'{base}/train/sequences.npy')

# Take the first 200 samples
rep_dataset = X_train[:200]

# Save it where the conversion script expects it
np.save(f'{base}/rep_dataset.npy', rep_dataset)
print(f"Saved representative dataset: {rep_dataset.shape}")

In [ ]:
!python /kaggle/working/convert_tflite.py \
 --model /kaggle/working/models/tms_isl.h5 \
 --output /kaggle/working/models/tms_isl_int8.tflite \
 --quantize int8 \
 --representative_dataset /kaggle/working/data/processed/rep_dataset.npy

In [ ]:
!python /kaggle/working/evaluate.py \
 --model /kaggle/working/models/tms_isl_int8.tflite \
 --dataset /kaggle/working/data/processed \
 --report /kaggle/working/models/check2_report.json

In [ ]:
%%writefile /kaggle/working/train_02.py
"""
train.py
SignBridge ML — Model Training Script (with Class Weights)
"""

import argparse
import os
import sys
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from sklearn.utils import class_weight

# Add parent src to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))
from tms_model import build_tms_model


def load_dataset(data_dir: str):
    data_path = Path(data_dir)
    with open(data_path / 'label_map.json', 'r') as f:
        label_map = json.load(f)
    n_classes = len(label_map)
    
    splits = {}
    for split in ['train', 'val', 'test']:
        split_dir = data_path / split
        sequences = np.load(split_dir / 'sequences.npy')
        labels = np.load(split_dir / 'labels.npy')
        
        # One-hot encode labels
        labels_onehot = keras.utils.to_categorical(labels, num_classes=n_classes)
        splits[split] = (sequences, labels_onehot)
        print(f"  {split}: sequences={sequences.shape}, labels={labels_onehot.shape}")
    
    return splits, label_map, n_classes


def augment_data(sequences: np.ndarray, labels: np.ndarray,
                 rotation_range: float = 15.0, time_warp: bool = True):
    augmented_seqs = []
    augmented_labels = []
    
    for seq, label in zip(sequences, labels):
        # Original
        augmented_seqs.append(seq)
        augmented_labels.append(label)
        
        # Augmented copy 1: rotation
        aug1 = seq.copy()
        angle = np.random.uniform(-rotation_range, rotation_range) * np.pi / 180
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        
        for t in range(len(aug1)):
            for i in range(0, 63, 3):
                x, y = aug1[t, i], aug1[t, i+1]
                aug1[t, i] = x * cos_a - y * sin_a
                aug1[t, i+1] = x * sin_a + y * cos_a
            for i in range(63, 126, 3):
                x, y = aug1[t, i], aug1[t, i+1]
                aug1[t, i] = x * cos_a - y * sin_a
                aug1[t, i+1] = x * sin_a + y * cos_a
        
        augmented_seqs.append(aug1)
        augmented_labels.append(label)
        
        # Augmented copy 2: noise
        aug2 = seq + np.random.randn(*seq.shape).astype(np.float32) * 0.02
        augmented_seqs.append(aug2)
        augmented_labels.append(label)
    
    return np.array(augmented_seqs), np.array(augmented_labels)


def train(args):
    print("="*60)
    print("  SignBridge TMS-Attention Model Training")
    print("="*60)
    
    splits, label_map, n_classes = load_dataset(args.dataset)
    X_train, y_train = splits['train']
    X_val, y_val = splits['val']
    X_test, y_test = splits['test']
    
    seq_len = X_train.shape[1]
    n_features = X_train.shape[2]
    
    if args.augment:
        print("\nApplying data augmentation...")
        X_train, y_train = augment_data(X_train, y_train)
        print(f"  After augmentation: {X_train.shape}")
        
    # --- NEW: Calculate Class Weights ---
    print("\nCalculating class weights for imbalanced data...")
    y_train_classes = np.argmax(y_train, axis=1)
    unique_classes = np.unique(y_train_classes)
    
    weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=unique_classes,
        y=y_train_classes
    )
    class_weight_dict = dict(zip(unique_classes, weights))
    print(f"  Weight range: Min = {min(weights):.2f}, Max = {max(weights):.2f}")
    # ------------------------------------

    print("\nBuilding TMS-Attention model...")
    model = build_tms_model(
        sequence_length=seq_len, n_features=n_features,
        n_classes=n_classes, d_model=128, num_heads=4,
        dropout_rate=0.3, label_smoothing=args.label_smoothing
    )
    
    if args.learning_rate != 0.001:
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=args.label_smoothing),
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
        )
    
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=args.early_stopping_patience, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        keras.callbacks.ModelCheckpoint(str(output_path), monitor='val_accuracy', save_best_only=True, verbose=1),
    ]
    
    print(f"\nTraining for up to {args.epochs} epochs...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=callbacks,
        class_weight=class_weight_dict,  # <-- NEW: Pass weights here
        verbose=1
    )
    
    print("\nEvaluating on test set...")
    test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"\n{'='*60}")
    print(f"  Training Complete")
    print(f"{'='*60}")
    print(f"  Best val accuracy: {max(history.history['val_accuracy']):.4f}")
    print(f"  Test accuracy:     {test_acc:.4f}")
    print(f"  Test top-5 acc:    {test_top5:.4f}")
    print(f"  Test loss:         {test_loss:.4f}")
    
    history_path = output_path.parent / f"{output_path.stem}_history.json"
    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(history_path, 'w') as f:
        json.dump(history_data, f, indent=2)
    
    return model, history

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', required=True)
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--learning_rate', type=float, default=0.001)
    parser.add_argument('--early_stopping_patience', type=int, default=10)
    parser.add_argument('--label_smoothing', type=float, default=0.1)
    parser.add_argument('--output', default='models/tms_wlasl100.h5')
    parser.add_argument('--augment', action='store_true', default=True)
    parser.add_argument('--no-augment', dest='augment', action='store_false')
    
    args = parser.parse_args()
    train(args)

In [ ]:
# 1. Train the new weighted model
!python /kaggle/working/train.py \
 --dataset /kaggle/working/data/processed \
 --epochs 100 \
 --batch_size 32 \
 --learning_rate 0.001 \
 --output /kaggle/working/models/tms_isl_weighted.h5

# 2. Convert to TFLite
!python /kaggle/working/convert_tflite.py \
 --model /kaggle/working/models/tms_isl_weighted.h5 \
 --output /kaggle/working/models/tms_isl_weighted_int8.tflite \
 --quantize int8 \
 --representative_dataset /kaggle/working/data/processed/rep_dataset.npy

# 3. Evaluate against CHECK-2 Thresholds
!python /kaggle/working/evaluate.py \
 --model /kaggle/working/models/tms_isl_weighted_int8.tflite \
 --dataset /kaggle/working/data/processed \
 --report /kaggle/working/models/check2_weighted_report.json

In [ ]:
%%writefile /kaggle/working/train_03.py
"""
train.py
SignBridge ML — Model Training Script (Targeted Augmentation + Low Dropout)
"""

import argparse
import os
import sys
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from sklearn.utils import class_weight

# Add parent src to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))
from tms_model import build_tms_model


def load_dataset(data_dir: str):
    data_path = Path(data_dir)
    with open(data_path / 'label_map.json', 'r') as f:
        label_map = json.load(f)
    n_classes = len(label_map)
    
    splits = {}
    for split in ['train', 'val', 'test']:
        split_dir = data_path / split
        sequences = np.load(split_dir / 'sequences.npy')
        labels = np.load(split_dir / 'labels.npy')
        
        labels_onehot = keras.utils.to_categorical(labels, num_classes=n_classes)
        splits[split] = (sequences, labels_onehot)
        print(f"  {split}: sequences={sequences.shape}, labels={labels_onehot.shape}")
    
    return splits, label_map, n_classes


def augment_data(sequences: np.ndarray, labels: np.ndarray, rotation_range: float = 15.0):
    """
    TARGETED AUGMENTATION: Upsamples rare classes to match the most common class.
    """
    augmented_seqs = list(sequences)
    augmented_labels = list(labels)
    
    y_classes = np.argmax(labels, axis=1)
    counts = np.bincount(y_classes, minlength=labels.shape[1])
    target_count = int(np.max(counts))  # Find the majority class count
    
    print(f"  Targeting {target_count} samples per class for perfect balance.")
    
    for c in range(labels.shape[1]):
        c_indices = np.where(y_classes == c)[0]
        if len(c_indices) == 0:
            continue
            
        num_needed = target_count - len(c_indices)
        if num_needed <= 0:
            continue
            
        # Generate exactly 'num_needed' copies to reach the target count
        for _ in range(num_needed):
            idx = np.random.choice(c_indices)
            seq = sequences[idx].copy()
            label = labels[idx]
            
            # Randomly pick an augmentation strategy for variety
            aug_type = np.random.choice(['rotate', 'noise', 'both'])
            
            if aug_type in ['rotate', 'both']:
                angle = np.random.uniform(-rotation_range, rotation_range) * np.pi / 180
                cos_a, sin_a = np.cos(angle), np.sin(angle)
                for t in range(len(seq)):
                    for i in range(0, 63, 3):
                        x, y = seq[t, i], seq[t, i+1]
                        seq[t, i] = x * cos_a - y * sin_a
                        seq[t, i+1] = x * sin_a + y * cos_a
                    for i in range(63, 126, 3):
                        x, y = seq[t, i], seq[t, i+1]
                        seq[t, i] = x * cos_a - y * sin_a
                        seq[t, i+1] = x * sin_a + y * cos_a
                        
            if aug_type in ['noise', 'both']:
                seq = seq + np.random.randn(*seq.shape).astype(np.float32) * 0.015
                
            augmented_seqs.append(seq)
            augmented_labels.append(label)
            
    return np.array(augmented_seqs), np.array(augmented_labels)


def train(args):
    print("="*60)
    print("  SignBridge TMS-Attention Model Training")
    print("="*60)
    
    splits, label_map, n_classes = load_dataset(args.dataset)
    X_train, y_train = splits['train']
    X_val, y_val = splits['val']
    X_test, y_test = splits['test']
    
    seq_len = X_train.shape[1]
    n_features = X_train.shape[2]
    
    if args.augment:
        print("\nApplying TARGETED data augmentation...")
        X_train, y_train = augment_data(X_train, y_train)
        print(f"  After augmentation: {X_train.shape}")
        
    print("\nCalculating class weights...")
    y_train_classes = np.argmax(y_train, axis=1)
    unique_classes = np.unique(y_train_classes)
    
    weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=unique_classes,
        y=y_train_classes
    )
    class_weight_dict = dict(zip(unique_classes, weights))

    print("\nBuilding TMS-Attention model...")
    model = build_tms_model(
        sequence_length=seq_len, n_features=n_features,
        n_classes=n_classes, d_model=128, num_heads=4,
        dropout_rate=0.15,  # <-- NEW: Lowered from 0.3 to 0.15
        label_smoothing=args.label_smoothing
    )
    
    if args.learning_rate != 0.001:
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=args.label_smoothing),
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
        )
    
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=args.early_stopping_patience, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        keras.callbacks.ModelCheckpoint(str(output_path), monitor='val_accuracy', save_best_only=True, verbose=1),
    ]
    
    print(f"\nTraining for up to {args.epochs} epochs...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    
    print("\nEvaluating on test set...")
    test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"\n{'='*60}")
    print(f"  Training Complete")
    print(f"{'='*60}")
    print(f"  Best val accuracy: {max(history.history['val_accuracy']):.4f}")
    print(f"  Test accuracy:     {test_acc:.4f}")
    print(f"  Test top-5 acc:    {test_top5:.4f}")
    
    history_path = output_path.parent / f"{output_path.stem}_history.json"
    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(history_path, 'w') as f:
        json.dump(history_data, f, indent=2)
    
    return model, history

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', required=True)
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--learning_rate', type=float, default=0.001)
    parser.add_argument('--early_stopping_patience', type=int, default=10)
    parser.add_argument('--label_smoothing', type=float, default=0.1)
    parser.add_argument('--output', default='models/tms_wlasl100.h5')
    parser.add_argument('--augment', action='store_true', default=True)
    parser.add_argument('--no-augment', dest='augment', action='store_false')
    
    args = parser.parse_args()
    train(args)

In [ ]:
# 1. Train the final model
!python /kaggle/working/train_03.py \
 --dataset /kaggle/working/data/processed \
 --epochs 100 \
 --batch_size 32 \
 --learning_rate 0.001 \
 --output /kaggle/working/models/tms_isl_final.h5

# 2. Convert to TFLite
!python /kaggle/working/convert_tflite.py \
 --model /kaggle/working/models/tms_isl_final.h5 \
 --output /kaggle/working/models/tms_isl_final_int8.tflite \
 --quantize int8 \
 --representative_dataset /kaggle/working/data/processed/rep_dataset.npy

# 3. Final CHECK-2 Evaluation
!python /kaggle/working/evaluate.py \
 --model /kaggle/working/models/tms_isl_final_int8.tflite \
 --dataset /kaggle/working/data/processed \
 --report /kaggle/working/models/check2_final_report.json

In [ ]:
%%writefile /kaggle/working/train_04.py
"""
train.py
SignBridge ML — Model Training Script (Focal Loss + Targeted Augmentation)
"""

import argparse
import os
import sys
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import tensorflow.keras.backend as K
from pathlib import Path
from sklearn.utils import class_weight

# Add parent src to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))
from tms_model import build_tms_model

def categorical_focal_loss(gamma=2.0, alpha=0.25):
    """
    Focal Loss: Punishes the model heavily for missing "hard/rare" examples,
    while relaxing the penalty for "easy/common" examples it already knows.
    """
    def focal_loss_fixed(y_true, y_pred):
        # Clip predictions to prevent math errors (NaNs)
        y_pred = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())
        # Standard cross-entropy
        cross_entropy = -y_true * K.log(y_pred)
        # Focal weight (gets larger if the model is unconfident)
        weight = alpha * K.pow(1.0 - y_pred, gamma)
        # Apply weight to loss
        loss = weight * cross_entropy
        return K.sum(loss, axis=-1)
    return focal_loss_fixed

def load_dataset(data_dir: str):
    data_path = Path(data_dir)
    with open(data_path / 'label_map.json', 'r') as f:
        label_map = json.load(f)
    n_classes = len(label_map)
    
    splits = {}
    for split in ['train', 'val', 'test']:
        split_dir = data_path / split
        sequences = np.load(split_dir / 'sequences.npy')
        labels = np.load(split_dir / 'labels.npy')
        
        labels_onehot = keras.utils.to_categorical(labels, num_classes=n_classes)
        splits[split] = (sequences, labels_onehot)
        print(f"  {split}: sequences={sequences.shape}, labels={labels_onehot.shape}")
    
    return splits, label_map, n_classes

def augment_data(sequences: np.ndarray, labels: np.ndarray, rotation_range: float = 15.0):
    """TARGETED AUGMENTATION: Upsamples rare classes to match the most common class."""
    augmented_seqs = list(sequences)
    augmented_labels = list(labels)
    
    y_classes = np.argmax(labels, axis=1)
    counts = np.bincount(y_classes, minlength=labels.shape[1])
    target_count = int(np.max(counts)) 
    
    print(f"  Targeting {target_count} samples per class for perfect balance.")
    
    for c in range(labels.shape[1]):
        c_indices = np.where(y_classes == c)[0]
        if len(c_indices) == 0:
            continue
            
        num_needed = target_count - len(c_indices)
        if num_needed <= 0:
            continue
            
        for _ in range(num_needed):
            idx = np.random.choice(c_indices)
            seq = sequences[idx].copy()
            label = labels[idx]
            
            aug_type = np.random.choice(['rotate', 'noise', 'both'])
            
            if aug_type in ['rotate', 'both']:
                angle = np.random.uniform(-rotation_range, rotation_range) * np.pi / 180
                cos_a, sin_a = np.cos(angle), np.sin(angle)
                for t in range(len(seq)):
                    for i in range(0, 63, 3):
                        x, y = seq[t, i], seq[t, i+1]
                        seq[t, i] = x * cos_a - y * sin_a
                        seq[t, i+1] = x * sin_a + y * cos_a
                    for i in range(63, 126, 3):
                        x, y = seq[t, i], seq[t, i+1]
                        seq[t, i] = x * cos_a - y * sin_a
                        seq[t, i+1] = x * sin_a + y * cos_a
                        
            if aug_type in ['noise', 'both']:
                seq = seq + np.random.randn(*seq.shape).astype(np.float32) * 0.015
                
            augmented_seqs.append(seq)
            augmented_labels.append(label)
            
    return np.array(augmented_seqs), np.array(augmented_labels)

def train(args):
    print("="*60)
    print("  SignBridge TMS-Attention Model Training (Focal Loss)")
    print("="*60)
    
    splits, label_map, n_classes = load_dataset(args.dataset)
    X_train, y_train = splits['train']
    X_val, y_val = splits['val']
    X_test, y_test = splits['test']
    
    seq_len = X_train.shape[1]
    n_features = X_train.shape[2]
    
    if args.augment:
        print("\nApplying TARGETED data augmentation...")
        X_train, y_train = augment_data(X_train, y_train)
        print(f"  After augmentation: {X_train.shape}")
        
    print("\nCalculating class weights...")
    y_train_classes = np.argmax(y_train, axis=1)
    unique_classes = np.unique(y_train_classes)
    
    weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=unique_classes,
        y=y_train_classes
    )
    class_weight_dict = dict(zip(unique_classes, weights))

    print("\nBuilding TMS-Attention model...")
    model = build_tms_model(
        sequence_length=seq_len, n_features=n_features,
        n_classes=n_classes, d_model=128, num_heads=4,
        dropout_rate=0.15, label_smoothing=args.label_smoothing
    )
    
    # --- NEW: Unconditionally apply Focal Loss ---
    print("Compiling model with Focal Loss (gamma=2.0)...")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate),
        loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
    )
    
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=args.early_stopping_patience, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        keras.callbacks.ModelCheckpoint(str(output_path), monitor='val_accuracy', save_best_only=True, verbose=1),
    ]
    
    print(f"\nTraining for up to {args.epochs} epochs...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    
    print("\nEvaluating on test set...")
    test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"\n{'='*60}")
    print(f"  Training Complete")
    print(f"{'='*60}")
    print(f"  Best val accuracy: {max(history.history['val_accuracy']):.4f}")
    print(f"  Test accuracy:     {test_acc:.4f}")
    print(f"  Test top-5 acc:    {test_top5:.4f}")
    
    history_path = output_path.parent / f"{output_path.stem}_history.json"
    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(history_path, 'w') as f:
        json.dump(history_data, f, indent=2)
    
    return model, history

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', required=True)
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--learning_rate', type=float, default=0.001)
    parser.add_argument('--early_stopping_patience', type=int, default=10)
    parser.add_argument('--label_smoothing', type=float, default=0.1)
    parser.add_argument('--output', default='models/tms_wlasl100.h5')
    parser.add_argument('--augment', action='store_true', default=True)
    parser.add_argument('--no-augment', dest='augment', action='store_false')
    
    args = parser.parse_args()
    train(args)

In [ ]:
# 1. Train the Focal Loss model
!python /kaggle/working/train_04.py \
 --dataset /kaggle/working/data/processed \
 --epochs 100 \
 --batch_size 32 \
 --learning_rate 0.001 \
 --output /kaggle/working/models/tms_isl_focal.h5

# 2. Convert to TFLite
!python /kaggle/working/convert_tflite.py \
 --model /kaggle/working/models/tms_isl_focal.h5 \
 --output /kaggle/working/models/tms_isl_focal_int8.tflite \
 --quantize int8 \
 --representative_dataset /kaggle/working/data/processed/rep_dataset.npy

# 3. Final CHECK-2 Evaluation
!python /kaggle/working/evaluate.py \
 --model /kaggle/working/models/tms_isl_focal_int8.tflite \
 --dataset /kaggle/working/data/processed \
 --report /kaggle/working/models/check2_focal_report.json

In [ ]:
# 1. Train the Focal Loss model
!python /kaggle/working/train_02.py \
 --dataset /kaggle/working/data/processed \
 --epochs 100 \
 --batch_size 32 \
 --learning_rate 0.001 \
 --output /kaggle/working/models/tms_isl_final_final.h5

# 2. Convert to TFLite
!python /kaggle/working/convert_tflite.py \
 --model /kaggle/working/models/tms_isl_final_final.h5 \
 --output /kaggle/working/models/tms_isl_final_final_int8.tflite \
 --quantize int8 \
 --representative_dataset /kaggle/working/data/processed/rep_dataset.npy

# 3. Final CHECK-2 Evaluation
!python /kaggle/working/evaluate.py \
 --model /kaggle/working/models/tms_isl_final_final_int8.tflite \
 --dataset /kaggle/working/data/processed \
 --report /kaggle/working/models/check2_final_final_report.json

In [ ]:
import numpy as np
import json

base = '/kaggle/working/data/processed'

# Load the integer labels from your training and test sets
y_train = np.load(f'{base}/train/labels.npy')
y_test = np.load(f'{base}/test/labels.npy')

# Load the label map to get the real sign names
with open(f'{base}/label_map.json', 'r') as f:
    label_map = json.load(f)

# Invert label map to easily look up names by their ID number
id_to_name = {v: k for k, v in label_map.items()}
n_classes = len(label_map)

print("=== Data Deficit Analysis ===")
print(f"Total classes expected: {n_classes}\n")

missing_in_train = []
for c in range(n_classes):
    train_count = np.sum(y_train == c)
    if train_count <= 4:
        missing_in_train.append(c)

print(f"Classes with atmost four training examples: {len(missing_in_train)}")
print("-" * 40)

for c in missing_in_train:
    test_count = np.sum(y_test == c)
    sign_name = id_to_name.get(c, f"Class {c}")
    print(f"Sign: {sign_name:<20} | In Train: 0 | In Test: {test_count}")

print("-" * 40)
print("Conclusion: The model cannot predict these signs because it never saw them during training.")
print("This permanently caps the maximum possible Macro-F1 score.")

In [ ]:
# Zip the models, the source code, and your python scripts into one file
!zip -r /kaggle/working/SignBridge_Outputs.zip \
  /kaggle/working/models/ \
  /kaggle/working/src/ \
  /kaggle/working/*.py

print("Zip file created successfully!")

In [ ]:
!zip -u /kaggle/working/SignBridge_Outputs.zip /kaggle/working/data/processed/label_map.json